# **Задание №9. Стратегия тестирования и тест-кейсы для системы сегментации ДЗЗ на основе HyenaPixel**

## **Введение**

Тестирование программного обеспечения — ключевой этап жизненного цикла разработки, обеспечивающий соответствие системы заявленным требованиям. Согласно исследованиям Capers Jones, стоимость исправления дефекта, обнаруженного на этапе эксплуатации, в 100 раз превышает стоимость его устранения на этапе проектирования.

В данном задании разрабатывается стратегия тестирования системы глубокого обучения для сегментации изображений дистанционного зондирования Земли (ДЗЗ) на основе архитектуры HyenaPixel. Основной акцент сделан на модуле инференса — наиболее критичном с точки зрения качества и производительности компоненте.

## **Формулировка задания**

На основе ранее разработанной спецификации требований создайте стратегию тестирования и набор тест-кейсов для системы сегментации ДЗЗ. **Необходимо выполнить:**

1. **Разработать стратегию тестирования** — виды, инструменты, метрики, критерии входа/выхода
2. **Создать функциональные тест-кейсы (≥ 15–20)** — акцент на модуле инференса
3. **Разработать нефункциональные тест-кейсы (≥ 5)** — производительность инференса, точность сегментации
4. **Построить матрицу трассируемости** — связь требований и тест-кейсов
5. **Определить метрики тестирования** — IoU, F1, время инференса, покрытие кода
6. **Подготовить план тестирования** — последовательность, сроки, ответственные

## **Теория: основы тестирования ПО**

### **1. Верификация и валидация**

**Тестирование** преследует две цели:

- **Верификация** — проверяет, что система построена *правильно* (соответствие требованиям)
- **Валидация** — проверяет, что построена *правильная* система (соответствие ожиданиям пользователей)

Для системы ML-инференса это особенно важно: модель может технически выполнять инференс (верификация), но при этом давать неприемлемое качество сегментации (валидация провалена).

### **2. Уровни тестирования**

| Уровень | Что тестируется | Инструменты | Пример для HyenaPixel |
|---------|----------------|-------------|----------------------|
| **Unit** | Отдельные функции/классы | pytest | `HyenaOperator.forward()`, нарезка тайлов |
| **Интеграционное** | Взаимодействие компонентов | pytest + fixtures | `InferenceEngine` + `ModelRegistry` + `DataService` |
| **Системное** | Вся система | pytest-e2e, MLflow | Полный пайплайн: загрузка сцены → инференс → маска |
| **Приёмочное** | Соответствие требованиям заказчика | ручное | Аналитик ДЗЗ оценивает качество сегментации |

### **3. Виды тестирования по знанию системы**

- **Чёрный ящик (Black Box)** — тестирование без знания внутренней структуры, по спецификации. Применяется при приёмочном тестировании результатов инференса.
- **Белый ящик (White Box)** — тестирование с полным знанием кода. Применяется при unit-тестировании операторов HyenaPixel и функций постобработки.
- **Серый ящик (Grey Box)** — комбинированный подход. Используется при интеграционном тестировании пайплайна.

### **4. Техники проектирования тест-кейсов**

#### **4.1. Эквивалентное разбиение (Equivalence Partitioning)**

**Принцип:** Входные данные делятся на классы, внутри которых система ведёт себя одинаково.

**Пример для инференса HyenaPixel — размер входной сцены:**

| Класс | Диапазон | Тест-значение | Ожидание |
|-------|---------|---------------|----------|
| Невалидный (мал) | < 64×64 пкс | 32×32 | Ошибка валидации |
| Валидный (стандарт) | 64×64 — 10000×10000 | 512×512 | Успешный инференс |
| Валидный (большой) | > 10000×10000 | 12000×12000 | Инференс с тайлингом |
| Невалидный (формат) | Неверный формат | `.docx`-файл | Ошибка формата |

#### **4.2. Граничные значения (Boundary Value Analysis)**

**Принцип:** Ошибки чаще возникают на границах классов эквивалентности.

**Пример для тайлинга сцен (размер тайла = 256 пкс):**

Граничные значения для тестирования:
- 255 пкс — ниже минимального тайла (невалидно)
- 256 пкс — минимальный тайл (граница, валидно)
- 257 пкс — чуть выше минимума (валидно)
- 512 пкс — стандартный тайл (валидно)
- 1023 пкс — перед максимумом (валидно)
- 1024 пкс — максимальный тайл (граница, валидно)
- 1025 пкс — превышение максимума (невалидно)

#### **4.3. Таблицы решений (Decision Tables)**

**Пример — выбор режима инференса в зависимости от размера сцены и доступности GPU:**

| № | Сцена > 2048 пкс | GPU доступен | Batch mode | Результат |
|---|-----------------|--------------|------------|-----------|
| 1 | Да | Да | Да | Тайлинг + GPU batch |
| 2 | Да | Да | Нет | Тайлинг + GPU single |
| 3 | Да | Нет | — | Тайлинг + CPU |
| 4 | Нет | Да | — | Прямой GPU-инференс |
| 5 | Нет | Нет | — | Прямой CPU-инференс |

#### **4.4. Диаграммы переходов состояний (State Transition Testing)**

**Пример — статусы задачи инференса (`InferenceRun.status`):**

```
QUEUED → RUNNING → COMPLETED
            ↓
          FAILED
            ↓
         CANCELLED  (из любого состояния, кроме COMPLETED)
```

Тест-кейсы проверяют:
- Допустимые переходы: `QUEUED → RUNNING`, `RUNNING → COMPLETED`
- Недопустимые переходы: `COMPLETED → RUNNING`, `FAILED → COMPLETED`
- Действия при переходах: сохранение маски при `COMPLETED`, запись ошибки при `FAILED`

## **Стратегия тестирования системы HyenaPixel**

### **1. Цели тестирования**

1. Убедиться, что модуль инференса корректно обрабатывает сцены ДЗЗ любых размеров
2. Проверить, что качество сегментации (IoU, F1) соответствует заявленным в SRS порогам
3. Верифицировать время инференса на стандартной GPU (NVIDIA A100) — не более 30 с/сцена (512×512)
4. Убедиться в корректности тайлинга, слияния тайлов и устранения артефактов на швах
5. Проверить устойчивость системы к некорректным входным данным

### **2. Область тестирования (Scope)**

**В область тестирования входит:**
- Модуль инференса: `InferenceEngine`, тайлинг, слияние, постобработка
- Загрузка и валидация входных сцен ДЗЗ
- Экспорт результатов сегментации (GeoTIFF, PNG)
- Вычисление метрик качества (IoU, F1) при наличии разметки
- Управление датасетами и сценами
- Обучение модели (SSL и supervised) — интеграционное тестирование

**Не входит в область тестирования:**
- Нагрузочное тестирование обучения (вне рамок дипломной работы)
- Тестирование сторонних библиотек (PyTorch, GDAL)

### **3. Виды тестирования**

| Вид тестирования | Применение | Инструмент | Приоритет |
|-----------------|------------|------------|----------|
| Unit-тестирование | Операторы HyenaPixel, функции тайлинга, метрики IoU/F1 | pytest | Высокий |
| Интеграционное | InferenceEngine + ModelRegistry + DataService | pytest + fixtures | Высокий |
| Системное (E2E) | Полный пайплайн инференса от загрузки до экспорта маски | pytest-e2e | Критический |
| Регрессионное | Проверка качества после обновления модели | pytest + MLflow | Высокий |
| Тестирование производительности | Время инференса, потребление GPU-памяти | pytest-benchmark | Высокий |
| Тестирование точности ML | IoU, F1 на тестовых датасетах | pytest + torchmetrics | Критический |
| Нагрузочное | Пакетная обработка нескольких сцен | locust / custom | Средний |
| Совместимости | Работа с разными форматами входных данных (GeoTIFF, PNG, JP2) | pytest | Средний |

### **4. Критерии входа и выхода**

**Критерии входа (Entry Criteria):**
- Код зафиксирован в системе контроля версий (Git)
- Тестовое окружение с GPU развёрнуто
- Тестовые датасеты ДЗЗ с ground truth масками подготовлены
- Чекпоинт обученной модели доступен

**Критерии выхода (Exit Criteria):**
- Все тест-кейсы с приоритетом «Критический» пройдены
- Покрытие требований ≥ 95%
- Mean IoU на тестовом датасете ≥ 0.75
- Среднее время инференса ≤ 30 с/сцена на GPU
- Pass Rate ≥ 98%
- Все критические дефекты исправлены

### **5. Инструменты тестирования**

| Инструмент | Назначение | Обоснование |
|------------|------------|-------------|
| **pytest** | Unit и интеграционное тестирование | Стандарт для Python-проектов, богатая экосистема |
| **pytest-benchmark** | Замер времени инференса | Встроен в pytest, статистически корректные замеры |
| **torchmetrics** | Вычисление IoU, F1, Dice | Официальная библиотека PyTorch для ML-метрик |
| **MLflow** | Логирование метрик экспериментов | Воспроизводимость, сравнение прогонов |
| **coverage.py** | Покрытие кода | Интеграция с pytest через pytest-cov |
| **hypothesis** | Property-based тестирование | Автогенерация граничных случаев для функций тайлинга |
| **GDAL / rasterio** | Валидация GeoTIFF на выходе | Проверка геопривязки и корректности растровых данных |

### **6. Окружение тестирования**

- **ОС:** Ubuntu 22.04 LTS
- **GPU:** NVIDIA A100 40GB (production) / NVIDIA RTX 3090 (dev)
- **Python:** 3.10+
- **PyTorch:** 2.x + CUDA 12.x
- **Тестовые данные:** датасет Potsdam (нарезанные сцены 512×512, 6 классов) + датасет Vaihingen
- **Тестовый стенд:** изолированная среда (conda env `test_hyenapixel`)

## **Функциональные тест-кейсы**

### **Модуль: Загрузка и валидация входных данных**

**TC-DATA-001: Успешная загрузка GeoTIFF-сцены с корректными метаданными**

| Поле | Значение |
|------|----------|
| **ID** | TC-DATA-001 |
| **Приоритет** | Критический |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-ДАТ-001: Система должна загружать сцены формата GeoTIFF |

**Предусловия:** Файл `test_scene.tif` (512×512, 4 канала, EPSG:32637) доступен в тестовой директории.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `DataService.load_scene('test_scene.tif')` | Объект `RemoteSensingScene` создан |
| 2 | Проверить атрибут `scene.width` | == 512 |
| 3 | Проверить атрибут `scene.height` | == 512 |
| 4 | Проверить атрибут `scene.bands` | == 4 |
| 5 | Проверить атрибут `scene.resolution` | Число > 0 |

**Постусловия:** Запись о сцене добавлена в таблицу `scenes`.

---

**TC-DATA-002: Отклонение файла неподдерживаемого формата**

| Поле | Значение |
|------|----------|
| **ID** | TC-DATA-002 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, негативное |

**Предусловия:** Файл `bad_file.docx` доступен.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `DataService.load_scene('bad_file.docx')` | Выброшено исключение `UnsupportedFormatError` |
| 2 | Проверить сообщение ошибки | Содержит название формата и список поддерживаемых |

---

**TC-DATA-003: Загрузка многоканальной сцены с нестандартным числом каналов**

| Поле | Значение |
|------|----------|
| **ID** | TC-DATA-003 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, граничное |

**Предусловия:** Файл `hyperspectral.tif` (1024×1024, 12 каналов).

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Загрузить файл через `DataService.load_scene()` | Успешная загрузка, `scene.bands == 12` |
| 2 | Попытаться запустить инференс (модель обучена на 4 каналах) | Выброшено `ChannelMismatchError` с описанием |


### **Модуль: Тайлинг сцен**

**TC-TILE-001: Корректная нарезка сцены на тайлы с перекрытием**

| Поле | Значение |
|------|----------|
| **ID** | TC-TILE-001 |
| **Приоритет** | Критический |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-ИНФ-002: Тайлинг должен поддерживать перекрытие (overlap) |

**Предусловия:** Сцена 1024×1024, размер тайла = 512, перекрытие = 64.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `InferenceEngine._tile_scene(scene, tile_size=512, overlap=64)` | Список тайлов |
| 2 | Проверить количество тайлов | == 4 (2×2 сетка с перекрытием) |
| 3 | Проверить размер каждого тайла | == (512, 512) |
| 4 | Проверить координаты тайлов | Учтено перекрытие 64 пкс |

---

**TC-TILE-002: Нарезка сцены, размер которой не кратен размеру тайла**

| Поле | Значение |
|------|----------|
| **ID** | TC-TILE-002 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, граничное |

**Предусловия:** Сцена 600×700, размер тайла = 512, перекрытие = 0.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить тайлинг | Тайлы созданы; крайние тайлы дополнены (padded) до 512×512 |
| 2 | Проверить, что padding отмечен | Атрибут `tile.is_padded == True` у крайних тайлов |
| 3 | После слияния масок проверить размер выходной маски | == 600×700 (padding обрезан) |

---

**TC-TILE-003: Нарезка сцены меньше размера тайла**

| Поле | Значение |
|------|----------|
| **ID** | TC-TILE-003 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, граничное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить тайлинг на сцене 64×64 с tile_size=512 | Один тайл с padding до 512×512 |
| 2 | После инференса и слияния | Маска обрезана обратно до 64×64 |

### **Модуль: Запуск инференса**

**TC-INF-001: Успешный инференс на одной сцене**

| Поле | Значение |
|------|----------|
| **ID** | TC-INF-001 |
| **Приоритет** | Критический |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-ИНФ-001: Система должна выполнять сегментацию загруженной сцены |

**Предусловия:** Сцена `potsdam_512.tif` загружена; модель `HyenaSegm_v1` зарегистрирована.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Создать `InferenceEngine(model='HyenaSegm_v1', tile_size=512)` | Объект создан, модель загружена на GPU |
| 2 | Вызвать `engine.run_on_scene(scene)` | Объект `InferenceResult` возвращён |
| 3 | Проверить `result.mask_path` | Файл существует на диске |
| 4 | Проверить размер маски | == (512, 512) |
| 5 | Проверить dtype маски | `uint8` (классовые индексы 0–N) |
| 6 | Проверить `result.status` в БД | == `completed` |

---

**TC-INF-002: Инференс на большой сцене с автоматическим тайлингом**

| Поле | Значение |
|------|----------|
| **ID** | TC-INF-002 |
| **Приоритет** | Критический |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-ИНФ-003: Сцены > 2048×2048 должны обрабатываться через тайлинг |

**Предусловия:** Сцена `large_scene_4096.tif` (4096×4096, 4 канала) загружена.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `engine.run_on_scene(scene)` | Автоматически активирован режим тайлинга |
| 2 | Проверить лог выполнения | Запись о разбивке на тайлы присутствует |
| 3 | Проверить размер результирующей маски | == (4096, 4096) |
| 4 | Проверить отсутствие видимых артефактов на швах тайлов | Маска непрерывна (проверка переходной зоны) |

---

**TC-INF-003: Инференс с несуществующим чекпоинтом модели**

| Поле | Значение |
|------|----------|
| **ID** | TC-INF-003 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, негативное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Создать `InferenceEngine(model='NonExistentModel')` | Выброшено `ModelNotFoundError` |
| 2 | Проверить статус `InferenceRun` в БД | == `failed`, поле `error_message` заполнено |

---

**TC-INF-004: Инференс при отсутствии GPU (CPU-fallback)**

| Поле | Значение |
|------|----------|
| **ID** | TC-INF-004 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс в окружении без CUDA | Система автоматически переключается на CPU |
| 2 | Проверить результат | Маска корректна (те же классы, что и на GPU) |
| 3 | Проверить предупреждение | В логе `WARNING: GPU unavailable, fallback to CPU` |

---

**TC-INF-005: Инференс на коррумпированном файле сцены**

| Поле | Значение |
|------|----------|
| **ID** | TC-INF-005 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, негативное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Передать файл с неполными данными (обрезанный GeoTIFF) | Выброшено `CorruptedSceneError` |
| 2 | Проверить статус задачи в БД | == `failed`, сообщение об ошибке информативно |

---

**TC-INF-006: Инференс с параметром `batch_size`**

| Поле | Значение |
|------|----------|
| **ID** | TC-INF-006 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить `engine.run_on_scene(scene, batch_size=8)` | Тайлы обрабатываются пакетами по 8 |
| 2 | Сравнить маску с результатом `batch_size=1` | Идентичные маски (детерминизм) |


### **Модуль: Слияние тайлов (Stitching)**

**TC-STITCH-001: Корректное слияние тайлов без артефактов**

| Поле | Значение |
|------|----------|
| **ID** | TC-STITCH-001 |
| **Приоритет** | Критический |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-ИНФ-004: Слияние тайлов должно применять сглаживание в зоне перекрытия |

**Предусловия:** 4 тайла 512×512 с перекрытием 64 пкс обработаны моделью.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `engine.stitch_tiles(tiles, overlap=64, method='weighted_average')` | Маска 1024×1024 |
| 2 | Проверить зону шва (строки 448–576) | Нет резких переходов классов, характерных для шва |
| 3 | Проверить отсутствие пустых (NaN/0) пикселей | Все пиксели имеют класс |

---

**TC-STITCH-002: Слияние тайлов методом majority voting**

| Поле | Значение |
|------|----------|
| **ID** | TC-STITCH-002 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `stitch_tiles(..., method='majority_voting')` | Маска создана |
| 2 | Сравнить IoU с методом `weighted_average` | Разница < 2% (оба метода приемлемы) |

### **Модуль: Экспорт результатов**

**TC-EXP-001: Экспорт маски сегментации в GeoTIFF с геопривязкой**

| Поле | Значение |
|------|----------|
| **ID** | TC-EXP-001 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-ЭКС-001: Маска должна экспортироваться в GeoTIFF с сохранением CRS исходной сцены |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `InferenceResult.export(format='geotiff', path='/tmp/mask.tif')` | Файл создан |
| 2 | Открыть файл через rasterio | Файл читается без ошибок |
| 3 | Проверить CRS | Совпадает с CRS исходной сцены (например, EPSG:32637) |
| 4 | Проверить трансформацию (affine transform) | Совпадает с исходной сценой |
| 5 | Проверить dtype | uint8 |

---

**TC-EXP-002: Экспорт маски в PNG (без геопривязки)**

| Поле | Значение |
|------|----------|
| **ID** | TC-EXP-002 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `InferenceResult.export(format='png', path='/tmp/mask.png')` | PNG-файл создан |
| 2 | Проверить размер изображения | Совпадает с размером сцены |
| 3 | Проверить режим изображения | `P` (palette mode) с кодами классов |

### **Модуль: Вычисление метрик качества**

**TC-METR-001: Корректный расчёт IoU при наличии ground truth**

| Поле | Значение |
|------|----------|
| **ID** | TC-METR-001 |
| **Приоритет** | Критический |
| **Тип** | Функциональное, позитивное |
| **Требование** | ФТ-МТР-001: Система должна рассчитывать mean IoU при наличии разметки |

**Предусловия:** Маска инференса и ground truth маска для сцены `potsdam_512.tif` доступны.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `MetricsCalculator.compute_iou(pred_mask, gt_mask, num_classes=6)` | Словарь IoU по классам |
| 2 | Проверить наличие ключей | Все 6 классов присутствуют |
| 3 | Проверить mean IoU | Число в диапазоне [0, 1] |
| 4 | Тест с идеальной маской (pred == gt) | mean IoU == 1.0 |
| 5 | Тест с полностью неверной маской | mean IoU == 0.0 |

---

**TC-METR-002: Расчёт F1-score (Dice) по классам**

| Поле | Значение |
|------|----------|
| **ID** | TC-METR-002 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `MetricsCalculator.compute_f1(pred_mask, gt_mask, num_classes=6)` | Словарь F1 по классам |
| 2 | Тест с идеальной маской | F1 == 1.0 для всех классов |
| 3 | Тест на маске с классом-пустышкой (не встречается в gt) | F1 == 0.0, не выброс исключения |

---

**TC-METR-003: Сохранение метрик в записи InferenceResult**

| Поле | Значение |
|------|----------|
| **ID** | TC-METR-003 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, интеграционное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс на сцене с разметкой | `InferenceResult.mean_iou` и `.mean_f1` заполнены |
| 2 | Запустить инференс на сцене без разметки | `.mean_iou == None`, `.mean_f1 == None` (не ошибка) |

### **Модуль: Управление датасетами**

**TC-DS-001: Создание датасета из набора сцен**

| Поле | Значение |
|------|----------|
| **ID** | TC-DS-001 |
| **Приоритет** | Высокий |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Создать датасет из 5 сцен через `DataService.create_dataset(scenes, name='test_ds')` | Датасет создан |
| 2 | Проверить связи в таблице `dataset_scenes` | 5 записей с корректными `scene_id` |
| 3 | Удалить одну сцену из датасета | Связь удалена, сам файл сцены не удалён |

---

**TC-DS-002: Разбивка датасета на train/val/test**

| Поле | Значение |
|------|----------|
| **ID** | TC-DS-002 |
| **Приоритет** | Средний |
| **Тип** | Функциональное, позитивное |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Вызвать `dataset.create_splits(train=0.7, val=0.15, test=0.15)` | Поле `split` у записей заполнено |
| 2 | Проверить отсутствие пересечений между сплитами | Одна сцена — ровно в одном сплите |
| 3 | Проверить сумму долей | Все сцены распределены (сумма == 1.0) |

## **Нефункциональные тест-кейсы**

### **Производительность инференса**

**TC-PERF-001: Время инференса одной сцены на GPU (≤ 30 с)**

| Поле | Значение |
|------|----------|
| **ID** | TC-PERF-001 |
| **Приоритет** | Высокий |
| **Тип** | Нефункциональное, производительность |
| **Требование** | НФТ-П-001: Инференс 512×512 сцены на GPU ≤ 30 с |

**Предусловия:** GPU NVIDIA A100 доступен; модель прогрета (1 warm-up прогон).

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс на 10 сценах 512×512 | Все прогоны завершены |
| 2 | Измерить среднее время (pytest-benchmark) | mean ≤ 30 с |
| 3 | Измерить максимальное время | max ≤ 45 с |

---

**TC-PERF-002: Потребление GPU-памяти при инференсе**

| Поле | Значение |
|------|----------|
| **ID** | TC-PERF-002 |
| **Приоритет** | Высокий |
| **Тип** | Нефункциональное, производительность |
| **Требование** | НФТ-П-002: Пиковое потребление VRAM ≤ 16 ГБ при tile_size=512 |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс; мониторинг через `torch.cuda.max_memory_allocated()` | Значение зафиксировано |
| 2 | Проверить пиковое потребление | ≤ 16 ГБ |
| 3 | Запустить пакетный инференс (batch_size=8) | Пиковое потребление ≤ 16 ГБ |

---

**TC-PERF-003: Масштабируемость тайлинга при увеличении размера сцены**

| Поле | Значение |
|------|----------|
| **ID** | TC-PERF-003 |
| **Приоритет** | Средний |
| **Тип** | Нефункциональное, производительность |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс на сценах 512, 1024, 2048, 4096 пкс | Все завершены |
| 2 | Построить зависимость время↔размер | Линейная зависимость (не квадратичная) |
| 3 | Проверить, что сцена 4096×4096 обрабатывается ≤ 5 минут | Время ≤ 300 с |

### **Точность сегментации (ML-качество)**

**TC-ACC-001: Mean IoU на тестовом датасете Potsdam ≥ 0.75**

| Поле | Значение |
|------|----------|
| **ID** | TC-ACC-001 |
| **Приоритет** | Критический |
| **Тип** | Нефункциональное, точность ML |
| **Требование** | НФТ-К-001: mean IoU ≥ 0.75 на датасете Potsdam |

**Предусловия:** 50 тестовых сцен из датасета Potsdam с ground truth масками.

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс на всех 50 сценах | Маски сохранены |
| 2 | Вычислить mean IoU по всем классам и сценам | mean IoU ≥ 0.75 |
| 3 | Проверить IoU для каждого класса отдельно | IoU ≥ 0.60 для каждого из 6 классов |

---

**TC-ACC-002: Регрессионный тест точности после обновления модели**

| Поле | Значение |
|------|----------|
| **ID** | TC-ACC-002 |
| **Приоритет** | Высокий |
| **Тип** | Нефункциональное, регрессия |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Загрузить новую версию модели `HyenaSegm_v2` | Модель доступна |
| 2 | Запустить инференс на фиксированном наборе из 20 сцен | Маски созданы |
| 3 | Сравнить mean IoU с `HyenaSegm_v1` | Деградация ≤ 2% (или улучшение) |
| 4 | Если деградация > 2% | Тест не пройден; требуется ревью модели |

### **Устойчивость и стабильность**

**TC-STAB-001: Детерминизм результатов инференса**

| Поле | Значение |
|------|----------|
| **ID** | TC-STAB-001 |
| **Приоритет** | Высокий |
| **Тип** | Нефункциональное, стабильность |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Запустить инференс одной сцены 3 раза подряд с одинаковыми параметрами | 3 маски созданы |
| 2 | Попиксельно сравнить маски | Все три маски идентичны |

---

**TC-STAB-002: Инференс при нехватке GPU-памяти (OOM)**

| Поле | Значение |
|------|----------|
| **ID** | TC-STAB-002 |
| **Приоритет** | Высокий |
| **Тип** | Нефункциональное, устойчивость |

| № | Действие | Ожидаемый результат |
|---|----------|---------------------|
| 1 | Смоделировать OOM (запустить сцену 8192×8192 без тайлинга) | Выброшено `OutOfMemoryError` |
| 2 | Проверить, что приложение не падает полностью | Исключение поймано, статус задачи == `failed` |
| 3 | Следующий запрос инференса с нормальной сценой | Выполняется успешно (GPU освобождена) |

## **Матрица трассируемости требований**

**Матрица трассируемости (RTM)** связывает требования из SRS с тест-кейсами, обеспечивая полноту покрытия.

| ID требования | Описание | Приоритет | Тест-кейсы | Покрытие |
|---------------|----------|-----------|------------|----------|
| ФТ-ДАТ-001 | Загрузка GeoTIFF-сцены | Критический | TC-DATA-001, TC-DATA-002, TC-DATA-003 | ✅ Покрыто |
| ФТ-ИНФ-001 | Сегментация одной сцены | Критический | TC-INF-001, TC-INF-003, TC-INF-005 | ✅ Покрыто |
| ФТ-ИНФ-002 | Тайлинг с перекрытием | Критический | TC-TILE-001, TC-TILE-002, TC-TILE-003 | ✅ Покрыто |
| ФТ-ИНФ-003 | Тайлинг для больших сцен | Критический | TC-INF-002, TC-PERF-003 | ✅ Покрыто |
| ФТ-ИНФ-004 | Слияние тайлов без артефактов | Критический | TC-STITCH-001, TC-STITCH-002 | ✅ Покрыто |
| ФТ-ИНФ-005 | CPU-fallback при отсутствии GPU | Средний | TC-INF-004 | ✅ Покрыто |
| ФТ-ИНФ-006 | Batch-инференс | Средний | TC-INF-006 | ✅ Покрыто |
| ФТ-ЭКС-001 | Экспорт в GeoTIFF с CRS | Высокий | TC-EXP-001 | ✅ Покрыто |
| ФТ-ЭКС-002 | Экспорт в PNG | Средний | TC-EXP-002 | ✅ Покрыто |
| ФТ-МТР-001 | Расчёт mean IoU | Критический | TC-METR-001, TC-METR-003 | ✅ Покрыто |
| ФТ-МТР-002 | Расчёт F1-score | Высокий | TC-METR-002, TC-METR-003 | ✅ Покрыто |
| ФТ-ДС-001 | Создание датасета | Высокий | TC-DS-001 | ✅ Покрыто |
| ФТ-ДС-002 | Разбивка на сплиты | Средний | TC-DS-002 | ✅ Покрыто |
| НФТ-П-001 | Время инференса ≤ 30 с/сцена | Высокий | TC-PERF-001 | ✅ Покрыто |
| НФТ-П-002 | VRAM ≤ 16 ГБ | Высокий | TC-PERF-002 | ✅ Покрыто |
| НФТ-К-001 | mean IoU ≥ 0.75 (Potsdam) | Критический | TC-ACC-001 | ✅ Покрыто |
| НФТ-К-002 | Регрессия точности | Высокий | TC-ACC-002 | ✅ Покрыто |
| НФТ-С-001 | Детерминизм результатов | Высокий | TC-STAB-001 | ✅ Покрыто |
| НФТ-С-002 | Устойчивость к OOM | Высокий | TC-STAB-002 | ✅ Покрыто |

## **Метрики тестирования**

### **Метрики покрытия**

$$Coverage_{req} = \frac{Requirements_{tested}}{Requirements_{total}} \times 100\%$$

Целевые значения:
- Критические требования: 100%
- Высокоприоритетные: ≥ 95%
- Средний приоритет: ≥ 80%

$$Coverage_{code} = \frac{Lines_{executed}}{Lines_{total}} \times 100\%$$

Целевые значения по уровням:
- Unit-тесты (inference_engine, metrics): ≥ 85%
- Интеграционные тесты: ≥ 70%
- Системные тесты (E2E): ≥ 60%

### **Метрики качества ML**

| Метрика | Формула | Целевое значение |
|---------|---------|------------------|
| **mean IoU** | $$(\frac{1}{C}\sum_{c=1}^{C}\frac{TP_c}{TP_c + FP_c + FN_c}$$ | ≥ 0.75 (Potsdam) |
| **mean F1 (Dice)** | $$(\frac{1}{C}\sum_{c=1}^{C}\frac{2 TP_c}{2 TP_c + FP_c + FN_c}$$ | ≥ 0.80 |
| **Per-class IoU** | IoU по каждому из C классов | ≥ 0.60 для каждого класса |
| **Pixel Accuracy** | $$(\frac{\sum TP_c}{\sum (TP_c + FP_c)}$$ | ≥ 0.85 |

### **Метрики выполнения тестов**

$$Pass\ Rate = \frac{Tests_{passed}}{Tests_{executed}} \times 100\%$$

Целевое значение перед релизом: ≥ 98%

$$DDE = \frac{Defects_{testing}}{Defects_{testing} + Defects_{production}} \times 100\%$$

Целевое значение: ≥ 95%

| Метрика | Цель |
|---------|------|
| Процент выполненных тестов | 100% (все запланированные) |
| Pass Rate (все тесты) | ≥ 98% |
| Pass Rate (критические) | 100% |
| Открытые критические дефекты | 0 перед релизом |
| Время инференса (mean, GPU) | ≤ 30 с |
| VRAM peak | ≤ 16 ГБ |

## **План тестирования**

### **Последовательность этапов**

| Этап | Вид тестирования | Что проверяется | Ответственный | Длительность |
|------|-----------------|-----------------|---------------|--------------|
| 1 | Unit-тестирование | Операторы HyenaPixel, тайлинг, метрики IoU/F1 | ML-инженер | 3 дня |
| 2 | Интеграционное | InferenceEngine + DataService + ModelRegistry | ML-инженер | 2 дня |
| 3 | Системное (E2E) | Полный пайплайн инференса | ML-инженер | 2 дня |
| 4 | ML-качество | IoU/F1 на Potsdam и Vaihingen | ML-инженер | 1 день |
| 5 | Производительность | Время инференса, VRAM | ML-инженер | 1 день |
| 6 | Регрессионное | После обновления модели | ML-инженер | 1 день |
| 7 | Приёмочное | Качество сегментации глазами аналитика | Аналитик ДЗЗ | 2 дня |

### **Процедура регистрации и обработки дефектов**

1. **Обнаружение** — тест-инженер фиксирует дефект с описанием шагов воспроизведения
2. **Классификация по серьёзности:**
   - *Critical* — инференс не выполняется, данные теряются
   - *Major* — значительное снижение IoU (> 10%), неверная маска
   - *Minor* — незначительные артефакты на швах тайлов
   - *Trivial* — косметические проблемы в логах
3. **Приоритизация** — Critical и Major блокируют релиз
4. **Исправление** — ML-инженер исправляет дефект, обновляет чекпоинт или код
5. **Регрессия** — повторный прогон тест-кейса + TC-ACC-002
6. **Закрытие** — тест-инженер подтверждает устранение